# Notebook 05 — Variograma empírico de residuales

Este notebook revisa si los modelos reducen la estructura espacial remanente en los residuales. La idea central es sencilla: si el componente espacial está bien capturado, el variograma de los residuales out-of-sample debería ser más plano y cercano al nugget que el variograma de la variable original.

Se comparan cuatro curvas: la variable simulada, los residuales al retirar únicamente la parte fija $\hat{F}(X)$, los residuales de GPBoost y los residuales de L2Boost. La comparación permite evaluar si modelar explícitamente el proceso espacial mejora frente a tratar las coordenadas como predictores ordinarios.


## Compatibilidad con Google Colab

La siguiente celda solo realiza acciones adicionales cuando el notebook se ejecuta en Google Colab. En ese caso, clona el repositorio para acceder a los archivos derivados ubicados en `data/`. En una ejecución local, asume que el notebook se corre desde la raíz del repositorio o desde la carpeta `notebooks/`.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/repo")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "-q", "https://github.com/linaq11/examen-2-gpboost.git", str(REPO_DIR)],
            check=True,
        )
    os.chdir(REPO_DIR)
else:
    current_dir = Path.cwd()
    if current_dir.name == "notebooks":
        os.chdir(current_dir.parent)

print(f"Directorio de trabajo: {Path.cwd()}")


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RANDOM_SEED = 1
DATA_DIR = Path("data")
FIGURE_DIR = Path("figuras")

FIGURE_DIR.mkdir(parents=True, exist_ok=True)
np.random.seed(RANDOM_SEED)


## 1. Carga de resultados precomputados

El cálculo de residuales y semivarianzas se realiza previamente para evitar reentrenar modelos en este notebook. Aquí solo se cargan las tablas necesarias para construir la figura diagnóstica y revisar los valores resumen.


In [ ]:
VARIOGRAM_PATH = DATA_DIR / "variograma_residuales.csv"
STATS_PATH = DATA_DIR / "variograma_stats.json"

required_files = [VARIOGRAM_PATH, STATS_PATH]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    raise FileNotFoundError(
        "No se encontraron los siguientes archivos requeridos:\n"
        + "\n".join(f"- {path}" for path in missing_files)
    )

variogram_df = pd.read_csv(VARIOGRAM_PATH)

required_columns = {
    "distancia",
    "gamma_y",
    "gamma_resid_F_only",
    "gamma_resid_gpboost",
    "gamma_resid_l2boost",
    "gamma_teorico_exp",
}

missing_columns = sorted(required_columns.difference(variogram_df.columns))
if missing_columns:
    raise ValueError(
        "La tabla de variograma no contiene las columnas requeridas: "
        + ", ".join(missing_columns)
    )

variogram_df.head()


## 2. Variograma de la variable original y de los residuales

La figura contrasta dos niveles de diagnóstico. El primer panel muestra la estructura espacial de la variable original y de los residuales que quedan al retirar solo la función fija $\hat{F}(X)$. El segundo panel compara los residuales out-of-sample de GPBoost y L2Boost.


In [ ]:
distance = variogram_df["distancia"].to_numpy()
gamma_y = variogram_df["gamma_y"].to_numpy()
gamma_fixed_only = variogram_df["gamma_resid_F_only"].to_numpy()
gamma_gpboost = variogram_df["gamma_resid_gpboost"].to_numpy()
gamma_l2boost = variogram_df["gamma_resid_l2boost"].to_numpy()
gamma_theoretical = variogram_df["gamma_teorico_exp"].to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(distance, gamma_y, "o-", label="y_train", color="black", linewidth=2)
axes[0].plot(
    distance,
    gamma_fixed_only,
    "s-",
    label="y - F̂(X), sin GP",
    color="darkgreen",
    linewidth=2,
)
axes[0].plot(
    distance,
    gamma_theoretical,
    "--",
    label="Teórico exponencial (σ²=0.35, ρ=0.1)",
    color="gray",
)
axes[0].axhline(
    0.45,
    color="red",
    linestyle=":",
    alpha=0.5,
    label="Sill teórico ≈ 0.45",
)
axes[0].set_xlabel("Distancia |sᵢ - sⱼ|")
axes[0].set_ylabel("γ(h)")
axes[0].set_title("a) Variable original y residuales sin GP")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].plot(
    distance,
    gamma_gpboost,
    "o-",
    color="steelblue",
    linewidth=2,
    label=f"GPBoost (γ̄={gamma_gpboost.mean():.3f})",
)
axes[1].plot(
    distance,
    gamma_l2boost,
    "s--",
    color="orange",
    linewidth=1.5,
    label=f"L2Boost (γ̄={gamma_l2boost.mean():.3f})",
)
axes[1].axhline(
    0.10,
    color="red",
    linestyle=":",
    alpha=0.5,
    label="Nugget teórico σ²=0.10",
)
axes[1].set_xlabel("Distancia |sᵢ - sⱼ|")
axes[1].set_ylabel("γ(h) de residuales")
axes[1].set_title("b) Residuales out-of-sample")
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

fig.tight_layout()

output_path = FIGURE_DIR / "fig8_variograma_residuales.png"
fig.savefig(output_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"Figura guardada en: {output_path}")


### Lectura metodológica

La variable original presenta una estructura espacial clara, visible en el aumento de la semivarianza con la distancia. Al retirar únicamente la función fija $\hat{F}(X)$, los residuales todavía conservan una señal espacial considerable, lo que indica que la parte no espacial del modelo no basta para absorber la dependencia inducida por $b(s)$.

En cambio, los residuales de GPBoost muestran una semivarianza media menor y más cercana al comportamiento esperado de un residuo sin estructura espacial fuerte. L2Boost también reduce la dependencia, pero su desempeño queda conceptualmente limitado porque incorpora las coordenadas como predictores ordinarios y no como un proceso espacial explícito.

La diferencia entre ambos enfoques respalda la conveniencia de modelar el componente espacial mediante una estructura de covarianza, especialmente cuando el objetivo es evaluar extrapolación o dependencia espacial residual.


## 3. Estadísticos resumen

Los estadísticos precomputados se cargan desde `variograma_stats.json`. Estos valores sirven como apoyo numérico para la interpretación de la figura.


In [ ]:
with STATS_PATH.open(encoding="utf-8") as file:
    variogram_stats = json.load(file)

print(json.dumps(variogram_stats, indent=2, ensure_ascii=False))
